<h1>Q-Learning</h1>

<p>In this exercise, you will examine the stochastic Q-Learning algorithm using the ready-made program provided to you.</p>
<p>Within the context of this example, a Q-learning implementation related to the autonomous driving system of a taxi will be presented. In this problem, the following requirements must be satisfied:</p>
<ul>
<li>The taxi must drop off the passenger at the correct location</li>
<li>The taxi must follow the shortest possible route</li>
<li>Traffic rules and passenger safety rules must be respected</li>
</ul>

<p>In the context of this problem, we have the following characteristics regarding rewards, states, and actions.</p>

<h4>Reward</h4>
<ul>
<li>We receive the maximum reward when the taxi drops off a passenger at the desired location</li>
<li>There is a penalty if the taxi drops the passenger off at an incorrect location</li>
<li>The agent receives a relatively small penalty if it is slow to reach the final destination</li>
</ul>

<p>In general, the above principles are summarized as follows: "We receive 20 points for a successful drop-off and lose 1 point for each time step taken. There is also a penalty of 10 points for illegal pickup and drop-off actions."</p>

<h4>Number of States</h4>
<img src="https://storage.googleapis.com/lds-media/images/Reinforcement_Learning_Taxi_Env.width-1200.png">
<p>In our example, we have a small 5*5 space. In addition, there are 4 destinations and 5 possible passenger locations (including the case where the passenger is already inside the taxi).</p>
<p>Based on the above, we have 5*5*5*4 = 500 possible states.</p>

<h4>Number of Actions</h4>
<p>We have six actions for the taxi, which are the following:</p>
<ul>
<li>0 = South</li>
<li>1 = North</li>
<li>2 = East</li>
<li>3 = West</li>
<li>4 = Pickup</li>
<li>5 = Dropoff</li>
</ul>





<p>Before continuing with the exercise, answer the following question:</p>

<b><p>1. Briefly describe the Q-Learning algorithm. In which problems do you think Reinforcement Learning is suitable as a learning approach? What is the main difference between the Q-Learning algorithm and the Policy Iteration and Value Iteration algorithms?</p></b>

<p>Next, you need to load the <i>gymnasium</i> library as well as the relevant dataset.</p>

In [ ]:
!pip install gymnasium

In [ ]:
import gymnasium as gym

env = gym.make("Taxi-v3", render_mode="rgb_array")
# Reset the environment to start
env.reset()

# Render the initial state
env.render()

In [ ]:
env.reset() # reset environment to a new, random state

print("Action Space {}".format(env.action_space))
print("State Space {}".format(env.observation_space))

# Render the initial state
env.render()

<p>Below, we define the taxi’s coordinates, the passenger’s location, and the destination point.</p>

In [ ]:
taxi = env.unwrapped            # the actual TaxiEnv


state = taxi.encode(3, 1, 2, 0) # (taxi row, taxi column, passenger index, destination index)
print("State:", state)

taxi.s = state

<p>Below is the reward matrix for the state we defined in the previous step.</p>

In [ ]:
taxi.P[328]

<p>We run our example without using Q-Learning.</p>

<b><p>2. Are the results satisfactory? How would the use of Q-Learning help us?</p></b>

In [ ]:
taxi.s = 328  # set environment to illustration's state

epochs = 0
penalties, reward = 0, 0

frames = [] # for animation

done = False

while not done:
    action = taxi.action_space.sample()
    state, reward, done, truncated, info = env.step(action)

    if reward == -10:
        penalties += 1

    # Put each rendered frame into dict for animation
    frames.append({
        'frame': taxi.render(),
        'state': state,
        'action': action,
        'reward': reward
        }
    )

    epochs += 1


print("Timesteps taken: {}".format(epochs))
print("Penalties incurred: {}".format(penalties))

<p>Now we will try to solve our problem using Q-Learning.</p>

<b><p>3. What do you know about the parameters α and γ? What would happen if they both had values equal to 1?</p></b>

In [7]:
import numpy as np
q_table = np.zeros([taxi.observation_space.n, taxi.action_space.n])

In [ ]:
%%time
"""Training the agent"""

import random
from IPython.display import clear_output

# Hyperparameters
alpha = 0.1
gamma = 0.6
epsilon = 0.1

# For plotting metrics
all_epochs = []
all_penalties = []

for i in range(1, 100001):
    state, _ = taxi.reset()

    epochs, penalties, reward, = 0, 0, 0
    done = False

    while not done:
        if random.uniform(0, 1) < epsilon:
            action = env.action_space.sample() # Explore action space
        else:
            action = np.argmax(q_table[state]) # Exploit learned values

        next_state, reward, done, truncated, info = taxi.step(action)

        old_value = q_table[state, action]
        next_max = np.max(q_table[next_state])

        new_value = (1 - alpha) * old_value + alpha * (reward + gamma * next_max)
        q_table[state, action] = new_value

        if reward == -10:
            penalties += 1

        state = next_state
        epochs += 1

    if i % 100 == 0:
        clear_output(wait=True)
        print(f"Episode: {i}")

print("Training finished.\n")

In [ ]:
q_table[328]

In [ ]:
"""Evaluate agent's performance after Q-learning"""

total_epochs, total_penalties = 0, 0
episodes = 100

for _ in range(episodes):
    state, _ = taxi.reset()
    epochs, penalties, reward = 0, 0, 0

    done = False

    while not done:
        action = np.argmax(q_table[state])
        state, reward, done, truncated, info = env.step(action)

        if reward == -10:
            penalties += 1

        epochs += 1

    total_penalties += penalties
    total_epochs += epochs

print(f"Results after {episodes} episodes:")
print(f"Average timesteps per episode: {total_epochs / episodes}")
print(f"Average penalties per episode: {total_penalties / episodes}")

<b><p>4. Compare the two algorithms based on the following metrics:</p>
<ul>
<li>Average number of penalties per episode</li>
<li>Average number of steps per trip</li>
<li>Average number of rewards per move</li>
</ul>
<p>Make the above comparisons over 100 episodes.</p>
</b>